## Comparaison Ovide / Bibles — proximité par embeddings SigLIP

Notebook autonome, extrait de `mini_rag_iconographique/vector_base.ipynb` : ne fait pas
partie du RAG (aucun de ses fichiers n'est lu par `rag_generation.ipynb` / `app_rag.py`),
c'est une analyse à part sur la proximité stylistique entre les deux corpus.

Un même espace de points : chaque illustration (Bible ou Ovide) est placée par projection
UMAP de son embedding SigLIP 768D. La couleur encode le thème (regroupé sur une taxonomie
commune aux deux corpus), la forme encode la source (cercle = Bible, triangle = Ovide).

**Prérequis** :
- `mini_rag_iconographique/vector_base.ipynb` déjà exécuté au moins une fois — il génère
  les deux fichiers lus ci-dessous (`bibles_siglip.pkl` et
  `ovide_bnu_corpus_comparaison_siglip.pkl`, dans `data/vector_bases/`).
- `lib/d3.v7.min.js` (à côté de ce notebook, gitignoré) — D3.js est embarqué directement
  dans le HTML généré plutôt que chargé depuis un CDN, pour que la page marche même sans
  connexion internet ou si le navigateur bloque les scripts distants sur une page ouverte
  en local (`file://`). Si absent, télécharger une fois :
  `curl -sL -o lib/d3.v7.min.js https://unpkg.com/d3@7/dist/d3.min.js`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

RACINE = Path("../../").resolve()
DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"


### 1. Chargement et taxonomie commune (regroupement des thèmes Bibles/Ovide)

In [ ]:
import umap

base_bibles = pd.read_pickle(DOSSIER_VECTOR_DB / "bibles_siglip.pkl")
base_ovide = pd.read_pickle(DOSSIER_VECTOR_DB / "ovide_bnu_corpus_comparaison_siglip.pkl")

# Taxonomie commune : les 12 classes fines des Bibles sont regroupées sous les 3
# thèmes partagés avec Ovide ; les classes sans équivalent Ovide restent groupées
# à part ("autre_bible") pour ne pas être comparées à tort.
MAPPING_THEME_BIBLE = {
    "Creation du MONDE": "creation_monde",
    "Creation HOMME": "creation_homme",
    "Creation EVE": "creation_homme",
    "Creation HORS ENTRAINEMENT": "creation_homme",
    "Deluge": "deluge",
    "Deluge AVANT": "deluge",
    "Deluge APRES": "deluge",
    "Babel": "autre_bible",
    "Cain et Abel": "autre_bible",
    "Chasses du paradis": "autre_bible",
    "Sacrifice ABRAHAM": "autre_bible",
    "Tentation": "autre_bible",
}

base_bibles = base_bibles.copy()
base_bibles["source"] = "bible"
base_bibles["theme_groupe"] = base_bibles["theme"].map(MAPPING_THEME_BIBLE)
base_bibles["theme_detail"] = base_bibles["theme"]
base_bibles["titre"] = np.nan  # pas de titre distinct du theme pour les Bibles

base_ovide = base_ovide.copy()
base_ovide["source"] = "ovide"
base_ovide["theme_groupe"] = base_ovide["theme"]
base_ovide["theme_detail"] = base_ovide["theme"]

points = pd.concat([
    base_bibles[["chemin", "source", "theme_groupe", "theme_detail", "titre", "embedding", "url_page", "url_image"]],
    base_ovide[["chemin", "source", "theme_groupe", "theme_detail", "titre", "embedding", "url_page", "url_image"]],
], ignore_index=True)

X_points = np.array(points["embedding"].tolist())
print(f"{len(points)} points -> embeddings {X_points.shape}")
print(points.groupby(["source", "theme_groupe"]).size())

### 2. Projection 2D (UMAP) et liens de plus proches voisins

In [3]:
# Projection 2D (UMAP, distance cosinus) pour la disposition visuelle
reducteur = umap.UMAP(n_neighbors=15, min_dist=0.3, metric="cosine", random_state=42)
coords = reducteur.fit_transform(X_points)
points["x"] = coords[:, 0]
points["y"] = coords[:, 1]

# Liens vers les plus proches voisins, calculés dans l'espace d'origine (768D),
# pas dans la projection 2D (qui déforme les distances).
sim_points = cosine_similarity(X_points)
np.fill_diagonal(sim_points, -1)

K_VOISINS = 3  # utilisé seulement pour la vérification trustworthiness plus bas
SEUIL_SIMILARITE = 0.95

# Un lien par paire dont la similarité dépasse le seuil (plus de plafond à k
# voisins par point : un point très central peut avoir beaucoup de liens, un
# point isolé peut n'en avoir aucun).
i_idx, j_idx = np.triu_indices_from(sim_points, k=1)
sims_paires = sim_points[i_idx, j_idx]
garder = sims_paires >= SEUIL_SIMILARITE

liens = [
    {"source": int(i), "target": int(j), "similarite": float(s)}
    for i, j, s in zip(i_idx[garder], j_idx[garder], sims_paires[garder])
]

print(f"{len(liens)} liens (seuil de similarité >= {SEUIL_SIMILARITE}, toutes les paires, sans plafond par point)")

/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


380 liens (seuil de similarité >= 0.95, toutes les paires, sans plafond par point)


### 3. Correspondances Ovide -> Bible (le thème est-il retrouvé ?)

In [4]:
# Correspondances Ovide -> Bible : pour chaque illustration Ovide, sa MEILLEURE
# correspondance Bible (n'importe quel theme), independamment du graphe de
# voisinage local ci-dessus. Sert a mesurer si l'iconographie Ovide "retombe"
# bien sur le meme theme cote Bible, ou sur un theme different.
ovide_idx = points.index[points["source"] == "ovide"].to_numpy()
bible_idx = points.index[points["source"] == "bible"].to_numpy()

X_ovide = X_points[ovide_idx]
X_bible = X_points[bible_idx]
sim_croisee = cosine_similarity(X_ovide, X_bible)

correspondances = []
for i, idx_o in enumerate(ovide_idx):
    j_meilleur = int(sim_croisee[i].argmax())
    idx_b = int(bible_idx[j_meilleur])
    correspondances.append({
        "ovide_id": int(idx_o),
        "bible_id": idx_b,
        "similarite": float(sim_croisee[i, j_meilleur]),
        "meme_theme": bool(points.loc[idx_o, "theme_groupe"] == points.loc[idx_b, "theme_groupe"]),
    })

df_correspondances = pd.DataFrame(correspondances)
df_correspondances["theme_ovide"] = df_correspondances["ovide_id"].map(points["theme_groupe"])

stats_theme = df_correspondances.groupby("theme_ovide").agg(
    n=("similarite", "size"),
    pct_meme_theme=("meme_theme", "mean"),
    similarite_moyenne=("similarite", "mean"),
).reset_index()

print("Correspondance Ovide -> meilleure Bible (n'importe quel theme), par theme Ovide :")
for _, r in stats_theme.iterrows():
    print(f"  {r['theme_ovide']:16s} : n={int(r['n']):2d}  meme theme={r['pct_meme_theme']:.0%}  similarite moyenne={r['similarite_moyenne']:.3f}")

# Correspondances Bible -> Ovide (sens inverse) : pour chaque illustration
# Bible, sa MEILLEURE correspondance Ovide - meme raisonnement, direction opposee.
correspondances_bible = []
for j, idx_b in enumerate(bible_idx):
    i_meilleur = int(sim_croisee[:, j].argmax())
    idx_o = int(ovide_idx[i_meilleur])
    correspondances_bible.append({
        "bible_id": int(idx_b),
        "ovide_id": idx_o,
        "similarite": float(sim_croisee[i_meilleur, j]),
        "meme_theme": bool(points.loc[idx_b, "theme_groupe"] == points.loc[idx_o, "theme_groupe"]),
    })

df_correspondances_bible = pd.DataFrame(correspondances_bible)
df_correspondances_bible["theme_bible"] = df_correspondances_bible["bible_id"].map(points["theme_groupe"])

# stats seulement sur les 3 themes partages ("autre_bible" n'a pas d'equivalent Ovide,
# la comparaison n'aurait pas de sens)
stats_theme_bible = (
    df_correspondances_bible[df_correspondances_bible["theme_bible"] != "autre_bible"]
    .groupby("theme_bible")
    .agg(
        n=("similarite", "size"),
        pct_meme_theme=("meme_theme", "mean"),
        similarite_moyenne=("similarite", "mean"),
    ).reset_index()
)

print("\nCorrespondance Bible -> meilleure Ovide (n'importe quel theme), par theme Bible (themes partages seulement) :")
for _, r in stats_theme_bible.iterrows():
    print(f"  {r['theme_bible']:16s} : n={int(r['n']):3d}  meme theme={r['pct_meme_theme']:.0%}  similarite moyenne={r['similarite_moyenne']:.3f}")

# Classements complets (pas seulement le 1er) -- necessaires cote navigateur pour
# recalculer "le meilleur voisin encore visible" quand un theme est filtre dans la
# legende, plutot que de laisser un point sans lien.
voisins_bible_vers_ovide = {}
for j, idx_b in enumerate(bible_idx):
    ordre = np.argsort(-sim_croisee[:, j])
    voisins_bible_vers_ovide[int(idx_b)] = [
        {"id": int(ovide_idx[i]), "similarite": float(sim_croisee[i, j])} for i in ordre
    ]

voisins_ovide_vers_bible = {}
for i, idx_o in enumerate(ovide_idx):
    ordre = np.argsort(-sim_croisee[i])
    voisins_ovide_vers_bible[int(idx_o)] = [
        {"id": int(bible_idx[j]), "similarite": float(sim_croisee[i, j])} for j in ordre
    ]

print(f"\nClassements complets exportes : {len(voisins_ovide_vers_bible)} points Ovide x "
      f"{len(bible_idx)} candidats Bible, {len(voisins_bible_vers_ovide)} points Bible x "
      f"{len(ovide_idx)} candidats Ovide.")

Correspondance Ovide -> meilleure Bible (n'importe quel theme), par theme Ovide :
  creation_homme   : n=17  meme theme=41%  similarite moyenne=0.895
  creation_monde   : n=25  meme theme=40%  similarite moyenne=0.878
  deluge           : n=31  meme theme=29%  similarite moyenne=0.893

Correspondance Bible -> meilleure Ovide (n'importe quel theme), par theme Bible (themes partages seulement) :
  creation_homme   : n= 56  meme theme=43%  similarite moyenne=0.862
  creation_monde   : n= 29  meme theme=62%  similarite moyenne=0.876
  deluge           : n= 65  meme theme=60%  similarite moyenne=0.852

Classements complets exportes : 73 points Ovide x 372 candidats Bible, 372 points Bible x 73 candidats Ovide.


### 4. Export du HTML interactif (données + gabarit)

In [5]:
import json as jsonlib

noeuds_json = []
for i, row in points.iterrows():
    noeuds_json.append({
        "id": int(i),
        "x": float(row["x"]),
        "y": float(row["y"]),
        "source": row["source"],
        "theme": row["theme_groupe"],
        "detail": row["theme_detail"],
        "titre": (str(row["titre"])[:90] if pd.notna(row["titre"]) else ""),
        "img": (row["url_image"] if pd.notna(row["url_image"]) else ""),
        "source_url": (row["url_page"] if pd.notna(row["url_page"]) else ""),
    })

stats_json = [
    {
        "theme": str(r["theme_ovide"]),
        "n": int(r["n"]),
        "pct_meme_theme": float(r["pct_meme_theme"]),
        "similarite_moyenne": float(r["similarite_moyenne"]),
    }
    for _, r in stats_theme.iterrows()
]

stats_bible_json = [
    {
        "theme": str(r["theme_bible"]),
        "n": int(r["n"]),
        "pct_meme_theme": float(r["pct_meme_theme"]),
        "similarite_moyenne": float(r["similarite_moyenne"]),
    }
    for _, r in stats_theme_bible.iterrows()
]

DONNEES_JSON = jsonlib.dumps({
    "noeuds": noeuds_json,
    "liens": liens,
    "correspondances": correspondances,
    "correspondances_bible": correspondances_bible,
    "voisinsOvideVersBible": voisins_ovide_vers_bible,
    "voisinsBibleVersOvide": voisins_bible_vers_ovide,
    "stats": stats_json,
    "stats_bible": stats_bible_json,
}, ensure_ascii=False)
print(f"{len(noeuds_json)} noeuds, {len(liens)} liens, {len(correspondances)} correspondances Ovide->Bible, "
      f"{len(correspondances_bible)} correspondances Bible->Ovide -> JSON pret ({len(DONNEES_JSON)/1024:.0f} Ko)")

445 noeuds, 380 liens, 73 correspondances Ovide->Bible, 372 correspondances Bible->Ovide -> JSON pret (2693 Ko)


In [6]:
HTML_TEMPLATE = """<!doctype html>
<html lang="fr">
<head>
<meta charset="utf-8">
<title>Proximite Bibles / Ovide - SigLIP</title>
<style>
  .viz-root {
    color-scheme: light;
    --surface-1:      #fcfcfb;
    --page-plane:     #f9f9f7;
    --text-primary:   #0b0b0b;
    --text-secondary: #52514e;
    --text-muted:     #898781;
    --gridline:       #e1e0d9;
    --border:         rgba(11,11,11,0.10);
    --series-1: #2a78d6; /* creation_monde */
    --series-2: #008300; /* creation_homme */
    --series-3: #e87ba4; /* deluge */
    --series-4: #eda100; /* autre_bible */
  }
  @media (prefers-color-scheme: dark) {
    :root:where(:not([data-theme="light"])) .viz-root {
      color-scheme: dark;
      --surface-1:      #1a1a19;
      --page-plane:     #0d0d0d;
      --text-primary:   #ffffff;
      --text-secondary: #c3c2b7;
      --text-muted:     #898781;
      --gridline:       #2c2c2a;
      --border:         rgba(255,255,255,0.10);
      --series-1: #3987e5;
      --series-2: #008300;
      --series-3: #d55181;
      --series-4: #c98500;
    }
  }
  :root[data-theme="dark"] .viz-root {
    color-scheme: dark;
    --surface-1:      #1a1a19;
    --page-plane:     #0d0d0d;
    --text-primary:   #ffffff;
    --text-secondary: #c3c2b7;
    --text-muted:     #898781;
    --gridline:       #2c2c2a;
    --border:         rgba(255,255,255,0.10);
    --series-1: #3987e5;
    --series-2: #008300;
    --series-3: #d55181;
    --series-4: #c98500;
  }
  * { box-sizing: border-box; }
  html, body { margin: 0; padding: 0; overflow-x: hidden; }
  body {
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
    background: var(--page-plane);
    color: var(--text-primary);
  }
  .viz-root { padding: 20px; }
  h1 { font-size: 1.15em; margin: 0 0 4px; }
  .sous-titre { color: var(--text-secondary); font-size: 0.85em; margin: 0 0 16px; }
  .cadre {
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 8px;
    position: relative;
    width: 100%;
    height: 78vh;
    min-height: 480px;
    overflow: hidden;
  }
  svg { width: 100%; height: 100%; display: block; cursor: grab; }
  svg:active { cursor: grabbing; }
  .lien { stroke: var(--text-muted); fill: none; }
  .marque { stroke: var(--border); stroke-width: 1px; cursor: pointer; }
  .marque.estompe { opacity: 0.08; }
  .marque.selectionne { stroke: var(--text-primary); stroke-width: 2.5px; }
  .legende {
    position: absolute; top: 12px; right: 12px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 10px 12px;
    font-size: 0.78em;
    color: var(--text-secondary);
    max-width: 220px;
  }
  .legende h4 { margin: 0 0 6px; font-size: 0.9em; color: var(--text-primary); font-weight: 600; }
  .legende-item { display: flex; align-items: center; gap: 6px; margin: 3px 0; cursor: pointer; user-select: none; }
  .legende-item.inactif { opacity: 0.35; }
  .legende-puce { width: 10px; height: 10px; border-radius: 50%; flex: none; }
  .legende-sep { height: 1px; background: var(--gridline); margin: 8px 0; }
  .legende-forme { width: 10px; height: 10px; flex: none; }
  .infobulle {
    position: absolute;
    pointer-events: none;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 8px;
    font-size: 0.78em;
    color: var(--text-primary);
    box-shadow: 0 4px 16px rgba(0,0,0,0.18);
    max-width: 380px;
    opacity: 0;
    transition: opacity 0.1s;
    z-index: 10;
  }
  .infobulle img { width: 100%; border-radius: 4px; margin-bottom: 6px; display: block; }
  .infobulle img.erreur { display: none; }
  .img-manquant {
    display: none;
    font-size: 0.85em;
    color: var(--text-muted);
    padding: 24px 0;
    text-align: center;
    background: var(--gridline);
    border-radius: 4px;
    margin-bottom: 6px;
  }
  .infobulle img.erreur + .img-manquant, .carte-comparaison img.erreur + .img-manquant { display: block; }
  .infobulle .theme { color: var(--text-secondary); }
  .infobulle.epingle { pointer-events: auto; }
  .infobulle .lien-voir {
    display: inline-block;
    margin-top: 6px;
    color: var(--text-primary);
    text-decoration: underline;
    font-weight: 600;
  }
  .infobulle .fermer-infobulle { margin-top: 6px; color: var(--text-muted); font-size: 0.9em; }
  .aide { color: var(--text-muted); font-size: 0.75em; margin-top: 8px; }
  .bascule {
    position: absolute; top: 12px; left: 12px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 4px;
    display: flex; gap: 4px;
    font-size: 0.78em;
  }
  .bascule button {
    font: inherit;
    border: none;
    background: transparent;
    color: var(--text-secondary);
    padding: 5px 9px;
    border-radius: 4px;
    cursor: pointer;
  }
  .bascule button.actif { background: var(--gridline); color: var(--text-primary); font-weight: 600; }
  .lien-correspondance.autreTheme { stroke-dasharray: 3,3; }
  .controles-bas {
    position: absolute; bottom: 12px; left: 12px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 8px 10px;
    font-size: 0.78em;
    color: var(--text-secondary);
    display: flex; align-items: center; gap: 12px; flex-wrap: wrap;
    max-width: 70%;
  }
  .controles-bas label { display: flex; align-items: center; gap: 6px; }
  .controles-bas input[type=range] { vertical-align: middle; }
  .controles-bas button {
    font: inherit; border: 1px solid var(--border); background: transparent;
    color: var(--text-secondary); padding: 4px 9px; border-radius: 4px; cursor: pointer;
  }
  .controles-bas button.actif { background: var(--gridline); color: var(--text-primary); font-weight: 600; }
  .plateau-comparaison {
    display: none;
    gap: 12px;
    margin-top: 14px;
    flex-wrap: wrap;
    align-items: flex-start;
  }
  .carte-comparaison {
    margin: 0;
    width: 220px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 8px;
  }
  .carte-comparaison img { width: 100%; border-radius: 4px; display: block; }
  .carte-comparaison figcaption { font-size: 0.78em; color: var(--text-secondary); margin-top: 6px; }
  .bouton-vider {
    align-self: center;
    font: inherit; border: 1px solid var(--border); background: transparent;
    color: var(--text-secondary); padding: 6px 10px; border-radius: 6px; cursor: pointer;
  }
  .panneau-stats {
    margin-top: 14px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 12px 16px;
  }
  .panneau-stats h3 { margin: 0 0 4px; font-size: 0.95em; }
  .panneau-stats p.note { color: var(--text-secondary); font-size: 0.78em; margin: 0 0 10px; }
  .panneau-stats table { border-collapse: collapse; width: 100%; font-size: 0.82em; }
  .panneau-stats th, .panneau-stats td { text-align: left; padding: 5px 10px; border-bottom: 1px solid var(--gridline); }
  .panneau-stats th { color: var(--text-muted); font-weight: 500; }
  .panneau-stats td.chiffre { font-variant-numeric: tabular-nums; }
</style>
</head>
<body>
<div class="viz-root">
  <h1>Proximite des illustrations - Bibles (classes_celine) vs Ovide (BNU_corpus)</h1>
  <p class="sous-titre">__N_NOEUDS__ illustrations, embeddings SigLIP (niveaux de gris) projetes en 2D (UMAP, distance cosinus). Cliquer la legende pour filtrer, cliquer un point pour n'afficher que ses liens, molette = zoom, glisser = deplacer.</p>
  <div class="cadre">
    <svg></svg>
    <div class="bascule" id="bascule">
      <button data-mode="local" class="actif">Voisinage local</button>
      <button data-mode="correspondance">Correspondances Ovide -&gt; Bible</button>
      <button data-mode="correspondance_bible">Correspondances Bible -&gt; Ovide</button>
    </div>
    <div class="legende" id="legende"></div>
    <div class="controles-bas" id="controles-bas">
      <label>Seuil de similarite : <span id="seuil-val"></span>
        <input type="range" id="seuil" step="0.001">
      </label>
      <button id="btn-comparaison" title="Cliquer jusqu'a 4 points pour les comparer en grand ci-dessous">Mode comparaison</button>
    </div>
    <div class="infobulle" id="infobulle"></div>
  </div>
  <p class="aide" id="aide-liens"></p>

  <div class="plateau-comparaison" id="plateau-comparaison"></div>

  <div class="panneau-stats">
    <h3>Ovide -&gt; meilleure correspondance Bible, par theme</h3>
    <p class="note">Pour chaque illustration Ovide, on cherche l'illustration Bible la plus
    proche (n'importe quel theme). "% meme theme" = a quelle frequence cette meilleure
    correspondance porte deja le meme theme cote Bible.</p>
    <table id="table-stats"></table>

    <h3 style="margin-top:16px;">Bible -&gt; meilleure correspondance Ovide, par theme</h3>
    <p class="note">Sens inverse : pour chaque illustration Bible d'un des 3 themes partages
    avec Ovide, sa meilleure correspondance Ovide. "Autre theme biblique" est exclu ici -
    Ovide n'a pas d'equivalent pour ces themes.</p>
    <table id="table-stats-bible"></table>
  </div>
</div>

<script>__D3_JS__</script>
<script>
const DONNEES = __DONNEES_JSON__;

const THEMES = [
  { cle: "creation_monde", label: "Creation du monde", var: "--series-1" },
  { cle: "creation_homme", label: "Creation de l'homme / Eve", var: "--series-2" },
  { cle: "deluge",         label: "Deluge",                     var: "--series-3" },
  { cle: "autre_bible",    label: "Autre theme biblique",       var: "--series-4" },
];

const racine = document.querySelector(".viz-root");
const style = getComputedStyle(racine);
const couleurTheme = {};
THEMES.forEach(t => couleurTheme[t.cle] = style.getPropertyValue(t.var).trim());

const cadre = document.querySelector(".cadre");
const svg = d3.select("svg");
const g = svg.append("g");
const gLiens = g.append("g").attr("class", "couche-liens");
const gNoeuds = g.append("g").attr("class", "couche-noeuds");

const infobulle = d3.select("#infobulle");
let infobullePinglee = false;

function ligneImage(url) {
  return `<img src="${url}" loading="lazy" class="img-vignette" onerror="this.classList.add('erreur')">
    <div class="img-manquant">Image indisponible</div>`;
}

function contenuInfobulle(d) {
  return `
    ${ligneImage(d.img)}
    <div><b>${d.source === "bible" ? "Bible" : "Ovide"}</b></div>
    <div class="theme">${d.detail}</div>
    ${d.titre ? `<div class="theme">${d.titre}</div>` : ""}
    ${d.source_url ? `<a class="lien-voir" href="${d.source_url}" target="_blank" rel="noopener">Voir la page source &#8599;</a>` : ""}
  `;
}

svg.on("click", () => {
  let quelqueChoseAChange = false;
  if (infobullePinglee) {
    infobullePinglee = false;
    infobulle.classed("epingle", false).style("opacity", 0);
    quelqueChoseAChange = true;
  }
  if (noeudIsoleId !== null) {
    noeudIsoleId = null;
    quelqueChoseAChange = true;
  }
  if (quelqueChoseAChange) dessinerLiens();
});

let largeur = cadre.clientWidth, hauteur = cadre.clientHeight;
let xEchelleActuelle = null, yEchelleActuelle = null;
// declares tot : dessiner() (appele plus bas) invoque dessinerLiens(), qui en depend
let modeLiens = "local";
let noeudIsoleId = null;       // id du noeud dont on isole les liens (clic simple), null = aucun
let modeComparaison = false;   // mode "cliquer jusqu'a 4 points pour les comparer"
let selectionComparaison = []; // ids selectionnes en mode comparaison
let seuilActuel = 0;           // seuil de similarite courant (0 = tout afficher), regle par reglerSeuilParDefaut()
const parIdIndex = new Map(DONNEES.noeuds.map(n => [n.id, n]));
const simCorr = d3.extent(DONNEES.correspondances, d => d.similarite);
const simCorrBible = d3.extent(DONNEES.correspondances_bible, d => d.similarite);
const simLocal = d3.extent(DONNEES.liens, d => d.similarite);

const formeParSource = {
  bible: d3.symbol().type(d3.symbolCircle).size(64),
  ovide: d3.symbol().type(d3.symbolTriangle).size(64),
};

// Legende unifiee : une entree par combinaison (source x theme) reellement
// presente dans les donnees, avec la forme utilisee dans le graphe (cercle
// Bible / triangle Ovide) et la couleur du theme. Clic = isoler ce groupe.
// Calcule tot (avant le premier dessiner()) : dessinerLiens() s'appuie sur
// combosActifs pour ne garder que les liens dont les DEUX extremites sont
// actuellement affichees.
const LABEL_THEME = Object.fromEntries(THEMES.map(t => [t.cle, t.label]));
const LABEL_SOURCE = { bible: "Bible", ovide: "Ovide" };

const groupesLegende = {};
DONNEES.noeuds.forEach(n => {
  const cle = n.source + "|" + n.theme;
  if (!groupesLegende[cle]) groupesLegende[cle] = { source: n.source, theme: n.theme };
});
const combosActifs = new Set(Object.keys(groupesLegende));

function noeudActif(id) {
  const n = parIdIndex.get(id);
  return !!n && combosActifs.has(n.source + "|" + n.theme);
}

function dessiner() {
  largeur = cadre.clientWidth;
  hauteur = cadre.clientHeight;
  svg.attr("viewBox", [0, 0, largeur, hauteur]);

  const xExtent = d3.extent(DONNEES.noeuds, d => d.x);
  const yExtent = d3.extent(DONNEES.noeuds, d => d.y);
  const marge = 40;
  const xScale = d3.scaleLinear().domain(xExtent).range([marge, largeur - marge]);
  const yScale = d3.scaleLinear().domain(yExtent).range([hauteur - marge, marge]);
  xEchelleActuelle = xScale;
  yEchelleActuelle = yScale;

  dessinerLiens();

  gNoeuds.selectAll("path")
    .data(DONNEES.noeuds)
    .join("path")
    .attr("class", "marque")
    .attr("transform", d => `translate(${xScale(d.x)},${yScale(d.y)})`)
    .attr("d", d => formeParSource[d.source]())
    .attr("fill", d => couleurTheme[d.theme] || "#999")
    .classed("selectionne", d => selectionComparaison.includes(d.id))
    .on("mouseenter", (evt, d) => {
      if (infobullePinglee) return;
      infobulle.style("opacity", 1).html(contenuInfobulle(d));
    })
    .on("mousemove", (evt) => {
      if (infobullePinglee) return;
      const [mx, my] = d3.pointer(evt, cadre);
      infobulle.style("left", (mx + 16) + "px").style("top", (my + 16) + "px");
    })
    .on("mouseleave", () => {
      if (infobullePinglee) return;
      infobulle.style("opacity", 0);
    })
    .on("click", (evt, d) => {
      evt.stopPropagation();

      if (modeComparaison) {
        const idx = selectionComparaison.indexOf(d.id);
        if (idx >= 0) selectionComparaison.splice(idx, 1);
        else if (selectionComparaison.length < 4) selectionComparaison.push(d.id);
        gNoeuds.selectAll("path").classed("selectionne", n => selectionComparaison.includes(n.id));
        majPlateauComparaison();
        return;
      }

      noeudIsoleId = (noeudIsoleId === d.id) ? null : d.id;
      dessinerLiens();

      const [mx, my] = d3.pointer(evt, cadre);
      infobullePinglee = true;
      infobulle
        .classed("epingle", true)
        .style("left", (mx + 16) + "px")
        .style("top", (my + 16) + "px")
        .style("opacity", 1)
        .html(contenuInfobulle(d) + `<div class="fermer-infobulle">Cliquer ailleurs pour fermer</div>`);
    });
}

dessiner();
window.addEventListener("resize", dessiner);

svg.call(d3.zoom().scaleExtent([0.3, 12]).on("zoom", (evt) => {
  g.attr("transform", evt.transform);
}));

const legende = d3.select("#legende");
["bible", "ovide"].forEach((source, i) => {
  const items = Object.values(groupesLegende).filter(g => g.source === source);
  if (!items.length) return;
  if (i > 0) legende.append("div").attr("class", "legende-sep");
  legende.append("h4").text(LABEL_SOURCE[source]);
  const sel = legende.selectAll(null)
    .data(items)
    .join("div")
    .attr("class", "legende-item")
    .on("click", (evt, d) => {
      const cle = d.source + "|" + d.theme;
      if (combosActifs.has(cle)) combosActifs.delete(cle); else combosActifs.add(cle);
      appliquerFiltre();
    });
  sel.append("svg").attr("class", "legende-forme").attr("viewBox", "-4 -4 8 8")
    .append("path")
    .attr("d", d => formeParSource[d.source]())
    .attr("fill", d => couleurTheme[d.theme]);
  sel.append("span").text(d => LABEL_THEME[d.theme] || d.theme);
});

function appliquerFiltre() {
  legende.selectAll(".legende-item").classed("inactif", d => !combosActifs.has(d.source + "|" + d.theme));
  gNoeuds.selectAll("path")
    .classed("estompe", d => !combosActifs.has(d.source + "|" + d.theme));
  dessinerLiens();
}

// Panneaux de statistiques par theme
const tableStats = d3.select("#table-stats");
tableStats.html(`
  <thead><tr><th>Theme (Ovide)</th><th>n</th><th>% meme theme cote Bible</th><th>Similarite moyenne</th></tr></thead>
  <tbody>
    ${DONNEES.stats.map(s => `
      <tr>
        <td>${LABEL_THEME[s.theme] || s.theme}</td>
        <td class="chiffre">${s.n}</td>
        <td class="chiffre">${(s.pct_meme_theme * 100).toFixed(0)}%</td>
        <td class="chiffre">${s.similarite_moyenne.toFixed(3)}</td>
      </tr>`).join("")}
  </tbody>
`);

const tableStatsBible = d3.select("#table-stats-bible");
tableStatsBible.html(`
  <thead><tr><th>Theme (Bible)</th><th>n</th><th>% meme theme cote Ovide</th><th>Similarite moyenne</th></tr></thead>
  <tbody>
    ${DONNEES.stats_bible.map(s => `
      <tr>
        <td>${LABEL_THEME[s.theme] || s.theme}</td>
        <td class="chiffre">${s.n}</td>
        <td class="chiffre">${(s.pct_meme_theme * 100).toFixed(0)}%</td>
        <td class="chiffre">${s.similarite_moyenne.toFixed(3)}</td>
      </tr>`).join("")}
  </tbody>
`);

// --- Mode comparaison -------------------------------------------------------
document.getElementById("btn-comparaison").addEventListener("click", function () {
  modeComparaison = !modeComparaison;
  this.classList.toggle("actif", modeComparaison);
  if (!modeComparaison) {
    selectionComparaison = [];
    gNoeuds.selectAll("path").classed("selectionne", false);
    majPlateauComparaison();
  }
});

function majPlateauComparaison() {
  const plateau = document.getElementById("plateau-comparaison");
  if (!selectionComparaison.length) { plateau.style.display = "none"; plateau.innerHTML = ""; return; }
  plateau.style.display = "flex";
  plateau.innerHTML = selectionComparaison.map(id => {
    const d = parIdIndex.get(id);
    return `<figure class="carte-comparaison">
      ${ligneImage(d.img)}
      <figcaption><b>${d.source === "bible" ? "Bible" : "Ovide"}</b> — ${d.detail}${d.titre ? " — " + d.titre : ""}</figcaption>
    </figure>`;
  }).join("") + `<button class="bouton-vider" onclick="viderComparaison()">Vider la selection</button>`;
}

function viderComparaison() {
  selectionComparaison = [];
  gNoeuds.selectAll("path").classed("selectionne", false);
  majPlateauComparaison();
}

// --- Seuil de similarite (filtre les liens deja calcules, cote client) ------
const seuilInput = document.getElementById("seuil");
const seuilVal = document.getElementById("seuil-val");

function extentPourMode() {
  if (modeLiens === "local") return simLocal;
  if (modeLiens === "correspondance") return simCorr;
  return simCorrBible;
}

function reglerSeuilParDefaut() {
  const [min, max] = extentPourMode();
  seuilInput.min = min;
  seuilInput.max = max;
  seuilInput.step = ((max - min) / 200) || 0.001;
  seuilInput.value = min;
  seuilActuel = min;
  seuilVal.textContent = min.toFixed(3);
}

seuilInput.addEventListener("input", () => {
  seuilActuel = parseFloat(seuilInput.value);
  seuilVal.textContent = seuilActuel.toFixed(3);
  dessinerLiens();
});

// Bascule entre liens de voisinage local (tous types de paires) et les deux
// sens de correspondance (Ovide -> meilleure Bible / Bible -> meilleure Ovide).
function majAide(nAffiches, nTotal) {
  let texte;
  if (modeLiens === "local") {
    texte = `Liens : ${nAffiches} / ${nTotal} paires affichees (seuil de similarite ajustable ci-dessous), calculee sur l'embedding 768D d'origine - toutes combinaisons Bible/Ovide confondues.`;
  } else if (modeLiens === "correspondance") {
    texte = `Liens : ${nAffiches} / ${nTotal} correspondances Ovide -> meilleure Bible affichees - trait plein = meme theme des deux cotes, pointille = theme different. Chaque illustration Ovide a toujours "une meilleure" Bible, bonne ou mauvaise : le seuil ci-dessous permet de ne garder que les correspondances vraiment fortes.`;
  } else {
    texte = `Liens : ${nAffiches} / ${nTotal} correspondances Bible -> meilleure Ovide affichees - trait plein = meme theme des deux cotes, pointille = theme different. Meme remarque que dans l'autre sens : ajuster le seuil pour ne garder que les correspondances fortes.`;
  }
  if (noeudIsoleId !== null) texte += " Un point est isole (clic ailleurs pour revenir a la vue complete).";
  d3.select("#aide-liens").text(texte);
}

function meilleurVoisinVisible(id, table) {
  const liste = table[id];
  if (!liste) return null;
  for (const cand of liste) {
    if (noeudActif(cand.id)) return cand;
  }
  return null;
}

// Pour chaque point actif du cote source, cherche son meilleur voisin encore
// visible de l'autre cote (recalcule si le 1er choix est filtre par la legende) --
// evite de laisser un point sans lien juste parce que sa meilleure correspondance
// a ete masquee.
function correspondancesDynamiquesOvideVersBible() {
  const resultats = [];
  DONNEES.noeuds.forEach(n => {
    if (n.source !== "ovide" || !noeudActif(n.id)) return;
    const meilleur = meilleurVoisinVisible(n.id, DONNEES.voisinsOvideVersBible);
    if (!meilleur) return;
    resultats.push({
      ovide_id: n.id,
      bible_id: meilleur.id,
      similarite: meilleur.similarite,
      meme_theme: parIdIndex.get(n.id).theme === parIdIndex.get(meilleur.id).theme,
    });
  });
  return resultats;
}

function correspondancesDynamiquesBibleVersOvide() {
  const resultats = [];
  DONNEES.noeuds.forEach(n => {
    if (n.source !== "bible" || !noeudActif(n.id)) return;
    const meilleur = meilleurVoisinVisible(n.id, DONNEES.voisinsBibleVersOvide);
    if (!meilleur) return;
    resultats.push({
      bible_id: n.id,
      ovide_id: meilleur.id,
      similarite: meilleur.similarite,
      meme_theme: parIdIndex.get(n.id).theme === parIdIndex.get(meilleur.id).theme,
    });
  });
  return resultats;
}

function correspondPourIsolement(d) {
  if (noeudIsoleId === null) return true;
  if (modeLiens === "local") return d.source === noeudIsoleId || d.target === noeudIsoleId;
  if (modeLiens === "correspondance") return d.ovide_id === noeudIsoleId || d.bible_id === noeudIsoleId;
  return d.bible_id === noeudIsoleId || d.ovide_id === noeudIsoleId;
}

function dessinerLiens() {
  gLiens.selectAll("line").remove();
  if (modeLiens === "local") {
    const donnees = DONNEES.liens.filter(d => d.similarite >= seuilActuel && correspondPourIsolement(d) && noeudActif(d.source) && noeudActif(d.target));
    gLiens.selectAll("line")
      .data(donnees)
      .join("line")
      .attr("class", "lien")
      .attr("x1", d => xEchelleActuelle(parIdIndex.get(d.source).x))
      .attr("y1", d => yEchelleActuelle(parIdIndex.get(d.source).y))
      .attr("x2", d => xEchelleActuelle(parIdIndex.get(d.target).x))
      .attr("y2", d => yEchelleActuelle(parIdIndex.get(d.target).y))
      .attr("stroke-width", 1)
      .attr("stroke-opacity", d => 0.05 + 0.35 * (d.similarite - simLocal[0]) / (simLocal[1] - simLocal[0] || 1));
    majAide(donnees.length, DONNEES.liens.length);
  } else if (modeLiens === "correspondance") {
    const brut = correspondancesDynamiquesOvideVersBible();
    const donnees = brut.filter(d => d.similarite >= seuilActuel && correspondPourIsolement(d));
    gLiens.selectAll("line")
      .data(donnees)
      .join("line")
      .attr("class", d => "lien lien-correspondance " + (d.meme_theme ? "memeTheme" : "autreTheme"))
      .attr("x1", d => xEchelleActuelle(parIdIndex.get(d.ovide_id).x))
      .attr("y1", d => yEchelleActuelle(parIdIndex.get(d.ovide_id).y))
      .attr("x2", d => xEchelleActuelle(parIdIndex.get(d.bible_id).x))
      .attr("y2", d => yEchelleActuelle(parIdIndex.get(d.bible_id).y))
      .attr("stroke-width", d => d.meme_theme ? 1.4 : 1)
      .attr("stroke-opacity", d => 0.2 + 0.6 * (d.similarite - simCorr[0]) / (simCorr[1] - simCorr[0] || 1));
    majAide(donnees.length, brut.length);
  } else {
    const brut = correspondancesDynamiquesBibleVersOvide();
    const donnees = brut.filter(d => d.similarite >= seuilActuel && correspondPourIsolement(d));
    gLiens.selectAll("line")
      .data(donnees)
      .join("line")
      .attr("class", d => "lien lien-correspondance " + (d.meme_theme ? "memeTheme" : "autreTheme"))
      .attr("x1", d => xEchelleActuelle(parIdIndex.get(d.bible_id).x))
      .attr("y1", d => yEchelleActuelle(parIdIndex.get(d.bible_id).y))
      .attr("x2", d => xEchelleActuelle(parIdIndex.get(d.ovide_id).x))
      .attr("y2", d => yEchelleActuelle(parIdIndex.get(d.ovide_id).y))
      .attr("stroke-width", d => d.meme_theme ? 1.4 : 1)
      .attr("stroke-opacity", d => 0.2 + 0.6 * (d.similarite - simCorrBible[0]) / (simCorrBible[1] - simCorrBible[0] || 1));
    majAide(donnees.length, brut.length);
  }
}

reglerSeuilParDefaut();
dessinerLiens();

d3.select("#bascule").selectAll("button").on("click", function (evt, d) {
  modeLiens = d3.select(this).attr("data-mode");
  d3.select("#bascule").selectAll("button").classed("actif", false);
  d3.select(this).classed("actif", true);
  noeudIsoleId = null;
  reglerSeuilParDefaut();
  dessinerLiens();
});
</script>
</body>
</html>"""

D3_JS = (Path(".") / "lib" / "d3.v7.min.js").read_text(encoding="utf-8")

html_final = (
    HTML_TEMPLATE
    .replace("__D3_JS__", D3_JS)
    .replace("__DONNEES_JSON__", DONNEES_JSON)
    .replace("__N_NOEUDS__", str(len(noeuds_json)))
)

DOSSIER_SORTIE = RACINE / "resultats" / "comparaison_ovide_bibles"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
chemin_sortie_viz = DOSSIER_SORTIE / "proximite_bibles_ovide.html"
chemin_sortie_viz.write_text(html_final, encoding="utf-8")
print(f"Visualisation generee : {chemin_sortie_viz}")
print(f"Taille : {chemin_sortie_viz.stat().st_size / 1024:.0f} Ko")

Visualisation generee : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/comparaison_ovide_bibles/proximite_bibles_ovide.html
Taille : 2991 Ko


### Vérification — la position 2D reflète-t-elle vraiment la proximité 768D ?

UMAP est construit pour préserver les voisinages locaux (contrairement à une
disposition aléatoire ou à un simple hasard de mise en page), mais on le mesure
plutôt que de le supposer :
- **Trustworthiness** (`sklearn.manifold.trustworthiness`) : parmi les k plus
  proches voisins de chaque point dans la projection 2D, quelle proportion sont
  aussi ses k plus proches voisins dans l'espace d'origine (768D) ? 1.0 = parfait,
  0.5 = pas mieux qu'aléatoire.
- **Longueur des liens** : les liens tracés relient déjà les vraies plus
  proches voisines (768D — nombre exact affiché juste au-dessus). On compare
  leur longueur en 2D à celle de paires aléatoires — si la projection est
  fidèle, les liens doivent être nettement plus courts que des paires prises
  au hasard.

In [7]:
from sklearn.manifold import trustworthiness

score_confiance = trustworthiness(X_points, coords, n_neighbors=K_VOISINS, metric="cosine")
print(f"Trustworthiness (k={K_VOISINS}) : {score_confiance:.3f}  (1.0 = parfait, 0.5 = aléatoire)")

# distances 2D des vraies paires plus-proches-voisines (les liens tracés)
dist_liens = np.array([
    np.linalg.norm([points.loc[l['source'], 'x'] - points.loc[l['target'], 'x'],
                     points.loc[l['source'], 'y'] - points.loc[l['target'], 'y']])
    for l in liens
])

# distances 2D de paires aléatoires (référence "si c'était sans rapport")
rng = np.random.default_rng(42)
n_ref = 2000
idx_a = rng.integers(0, len(points), n_ref)
idx_b = rng.integers(0, len(points), n_ref)
dist_aleatoire = np.array([
    np.linalg.norm([points.loc[a, 'x'] - points.loc[b, 'x'], points.loc[a, 'y'] - points.loc[b, 'y']])
    for a, b in zip(idx_a, idx_b) if a != b
])

print(f"\nDistance 2D moyenne — vraies plus proches voisines (768D) : {dist_liens.mean():.2f}")
print(f"Distance 2D moyenne — paires aléatoires                    : {dist_aleatoire.mean():.2f}")
print(f"Ratio (plus petit = mieux) : {dist_liens.mean() / dist_aleatoire.mean():.2f}")

Trustworthiness (k=3) : 0.931  (1.0 = parfait, 0.5 = aléatoire)



Distance 2D moyenne — vraies plus proches voisines (768D) : 0.33
Distance 2D moyenne — paires aléatoires                    : 3.86
Ratio (plus petit = mieux) : 0.09
